<a href="https://colab.research.google.com/github/Jumpr15/pytorch-work/blob/main/flappy_bird_reinforce_working.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install flappy-bird-gymnasium gymnasium[other]

In [2]:
import torch.distributions as distributions
import torch.nn as nn
import torch

class Policy:
  def __init__(self, model, optimizer):
    self.model = model
    self.optimizer = optimizer

  def sample_action(self, obs):
    probs = self.model(obs)
    distri = distributions.Categorical(probs=probs)
    action = distri.sample()
    return action.item()

  def discount_rewards(self, reward_tensor, discount_factor=0.99):
    discount_list = []
    reward_sum = 0

    for reward in reversed(reward_tensor):
        reward_sum = reward + (reward_sum * discount_factor)
        discount_list.append(reward_sum)

    discounted_tensor = torch.flip(torch.tensor(discount_list), dims=[0])
    final_rewards = (discounted_tensor - discounted_tensor.mean()) / (discounted_tensor.std() + 1e-8)
    return final_rewards

  def calculate_loss(self, train_states, train_actions, train_rewards):
    train_states = torch.tensor(train_states, dtype=torch.float32)
    train_actions = torch.tensor(train_actions, dtype=torch.float32)
    train_rewards = torch.tensor(train_rewards, dtype=torch.float32)

    discounted_rewards = self.discount_rewards(train_rewards)

    probs = self.model(train_states)
    distri = distributions.Categorical(probs=probs)
    loss = -distri.log_prob(train_actions) * discounted_rewards
    return loss

  def optimize_policy(self, train_states, train_actions, train_rewards):
    loss_tensor = self.calculate_loss(train_states, train_actions, train_rewards)
    loss = loss_tensor.mean()

    self.optimizer.zero_grad()
    loss.backward()

    nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=0.6)
    self.optimizer.step()

    return loss.item()

In [9]:
import torch.nn as nn
import torch.optim as optim

in_dims = 180
h_dims = 256
out_dims = 2
model = nn.Sequential(
    nn.Linear(in_dims, h_dims),
    nn.ReLU(),
    nn.Linear(h_dims, h_dims),
    nn.ReLU(),
    nn.Linear(h_dims, out_dims),
    nn.Softmax(dim=-1)
)

lr = 5e-5
optimizer = optim.Adam(
    model.parameters(),
    lr=lr
)

policy = Policy(model, optimizer)

In [13]:
# @title
import flappy_bird_gymnasium
import gymnasium as gym
from gymnasium.wrappers import RecordVideo

def record_step_trigger(step: int) -> bool:
  per_step = 100
  if step % per_step == 0:
    return True
  return False

env = RecordVideo(gym.make("FlappyBird-v0", render_mode="rgb_array", use_lidar=True), video_folder='./video-output-3')

In [14]:
def train_epoch(env):
  obs, _ = env.reset()

  train_states = []
  train_actions = []
  train_rewards = []

  while True:
    obs_tensor = (torch.from_numpy(obs)).to(torch.float32)
    action = policy.sample_action(obs_tensor)

    train_states.append(obs)
    train_actions.append(action)

    obs, reward, terminated, _, info = env.step(action)

    train_rewards.append(reward)

    if terminated:
      break

  env.close()
  return train_states, train_actions, train_rewards

In [ ]:
torch.manual_seed(67)
episodes = 5000
for _ in range(episodes):
  train_states, train_actions, train_rewards = train_epoch(env)
  loss = policy.optimize_policy(train_states, train_actions, train_rewards)
  print(loss)